In [15]:
import os
import sys

# Add the parent directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
from tqdm import tqdm
import pickle
from ml_attack.utils import cmod

data_path = "./../../salsa_data/80_7_omega15_lwe_data_prefix/"

A = np.load(data_path + "/origA_n80_logq7.npy")
with open(data_path + "/params.pkl", "rb") as f:
  loaded_params = pickle.load(f)

In [16]:
loaded_params

{'dump_path': './data/benchmark_paper_data/n80_logq7/',
 'exp_name': 'training_data',
 'exp_id': '',
 'tag': '',
 'log_every': 500,
 'seed': 42,
 'actions': ['secrets'],
 'processed_dump_path': './data/benchmark_paper_data/n80_logq7/',
 'num_secret_seeds': 10,
 'min_hamming': 5,
 'max_hamming': 6,
 'secret_type': 'binary',
 'sigma': 3,
 'gamma': 2,
 'max_samples': 2000000,
 'std_threshold': 1.0,
 'rlwe': 1,
 'nu': 54,
 'command': ('python src/generate/generate_secrets.py --dump_path ./data/benchmark_paper_data/n80_logq7 --processed_dump_path ./data/benchmark_paper_data/n80_logq7/ --secret_type binary --min_hamming 5 --max_hamming 6 --rlwe 1 --actions secrets --max_samples 2000000 --num_secret_seeds 10 --exp_name training_data',),
 'N': 80,
 'Q': 113,
 'logq': 7,
 'omega': 10,
 'bkz_block_size1': 18,
 'bkz_block_size2': 22,
 'write_threshold': 0.86101,
 'secret_dir': './data/benchmark_paper_data/n80_logq7/binary_secrets_h5_6',
 'm': -1,
 'orig_A_path': './data/benchmark_paper_data/n80_l

In [3]:
k = loaded_params["rlwe"]
n = A.shape[1] // k
num_gen = A.shape[0] // (n * k)
if "m" not in loaded_params or loaded_params["m"] == -1:
    m = loaded_params["N"]
else:
    m = loaded_params["m"]

q = loaded_params["Q"]

print(f"n = {n}, k = {k}, num_gen = {num_gen}, m = {m}, q = {q}")

n = 80, k = 1, num_gen = 4, m = 80, q = 113


In [ ]:
full_R = []
full_indices = []
max_samples = 10000
with open(data_path + "/data.prefix") as fd:
    indices, RT = [], []
    for line in tqdm(fd):
        if not line:
            continue

        ind, r = line.strip().split(";")
        indices.append(int(ind.strip()))
        RT.append(np.array(r.split(), dtype=np.int64))
        if len(indices) == m:
            full_indices.append(indices)
            full_R.append(np.array(RT).T)
            indices, RT = [], []
            max_samples -= 1
            if max_samples <= 0:
                break

full_R = np.stack(full_R)
full_indices = np.stack(full_indices)

799999it [00:21, 38066.07it/s]


In [5]:
full_R

array([[[ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        ...,
        [ 0, -1,  0, ...,  0,  1,  0],
        [ 0,  1,  0, ...,  0,  1,  0],
        [ 0, -1, -1, ...,  0,  0,  1]],

       [[ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        ...,
        [-1,  0,  0, ...,  0,  0, -1],
        [ 0, -1,  0, ...,  1,  0,  0],
        [ 0,  0,  1, ..., -1,  0,  0]],

       [[ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        ...,
        [ 1, -1,  0, ..., -1,  0, -1],
        [ 0, -1,  1, ...,  0,  0, -1],
        [-1,  2,  1, ...,  1, -2,  1]],

       ...,

       [[ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        [ 0,  0,  0, ...,  0,  0,  0],
        ...,
        [ 2,  0, -1, ...,  0,  0, -1],
        [-1, -2, -2, ..., -2,  0,  0],
        [ 0, -2, -2, ..., -1,  0

In [6]:
full_R.shape

(10000, 160, 80)

In [7]:
full_indices.shape

(10000, 80)

In [8]:
full_indices

array([[ 93,  63,  46, ..., 188, 245, 135],
       [ 90, 297, 181, ..., 104, 174,  17],
       [ 68, 316, 160, ...,  87, 142, 122],
       ...,
       [191,  92, 205, ...,  94, 265,  27],
       [ 89,  54,  79, ...,  53,  40,   1],
       [119, 111, 262, ..., 216,  14, 204]])

In [9]:
A_to_reduce = np.stack([A[ind] for ind in full_indices])

In [10]:
RA = cmod(full_R @ A_to_reduce, q)

In [11]:
RA

array([[[  0,   0,   0, ...,   0,   0,   0],
        [  0,   0,   0, ...,   0,   0,   0],
        [  0,   0,   0, ...,   0,   0,   0],
        ...,
        [-56,  -2, -38, ...,  -1,  19,  -3],
        [ 50, -20,  15, ...,  14,  -6,  -2],
        [-41,  54,  -4, ...,  11,   7, -32]],

       [[  0,   0,   0, ...,   0,   0,   0],
        [  0,   0,   0, ...,   0,   0,   0],
        [  0,   0,   0, ...,   0,   0,   0],
        ...,
        [ 30, -26,  15, ...,   2,  -3,   2],
        [ 11, -33,   7, ...,  -3,  19,  -2],
        [ 50, -54, -44, ...,   4, -13, -23]],

       [[  0,   0,   0, ...,   0,   0,   0],
        [  0,   0,   0, ...,   0,   0,   0],
        [  0,   0,   0, ...,   0,   0,   0],
        ...,
        [ 19,  14,  35, ...,  16,   1,  19],
        [-13,  18,  27, ...,   7,  32,  46],
        [  3, -40,   9, ..., -19,  -4, -12]],

       ...,

       [[  0,   0,   0, ...,   0,   0,   0],
        [  0,   0,   0, ...,   0,   0,   0],
        [  0,   0,   0, ...,   0,   0,   0

In [36]:
RA = RA[np.any(RA != 0, axis=-1)]
np.mean(np.std(RA, axis=-1))

np.float64(25.296059249692)

Automatic loading:

In [1]:
import os
import sys

# Add the parent directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from ml_attack.dataset import LWEDataset

data_path = "./../../salsa_data/n80_logq7/"

dataset = LWEDataset.load_reduced_from_salsa(data_path=data_path, top_percent=0.2)
dataset.params['verbose'] = True

In [2]:
dataset.RA.shape

(36719, 32, 80)

In [3]:
dataset.initialize_secret()
dataset.params['train_percentages'] = [0.005, 0.01, 1.0]
dataset.train()

[BEST 5% STD] True B is the best candidate: 20369 / 31887 (63.88%)
[BEST 5% STD] Expected true B is best candidate: 67.96%


Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


✔️ Patched scikit-learn (once).
[BEST 10% STD] True B is the best candidate: 39936 / 63774 (62.62%)
[BEST 10% STD] Expected true B is best candidate: 66.94%
[BEST 20% STD] True B is the best candidate: 78532 / 127548 (61.57%)
[BEST 20% STD] Expected true B is best candidate: 65.79%


(False, None)